In [1]:
import pandas as pd
import os, pathlib, getpass

In [2]:
USERNAME = getpass.getuser()
HOME_DIR = pathlib.Path.home()
M_DRIVE = pathlib.Path("/Volumes/Data/Models") if os.name != "nt" else pathlib.Path("M:/")
if USERNAME.lower() in ['lzorn', 'jahrenholtz']: # need to standardize to E:Box
    BOX_DIR = pathlib.Path("E:/Box")
elif USERNAME.lower() in ['aolsen']:
    BOX_DIR = HOME_DIR /'Library/CloudStorage/Box-Box'
elif USERNAME.lower() in ['ywang']:
    BOX_DIR = pathlib.Path("C:/Users/ywang/Box")

### get parcel data and parcel-geographies crosswalk

In [3]:
FBP_DIR = r"M:\urban_modeling\baus\PBA50Plus\PBA50Plus_FinalBlueprint\PBA50Plus_Final_Blueprint_v65"
NP_DIR = r"M:\urban_modeling\baus\PBA50Plus\PBA50Plus_NoProject\PBA50Plus_NoProject_v38"
FBP_RUNID = "PBA50Plus_Final_Blueprint_v65"
NP_RUNID = "PBA50Plus_NoProject_v38"

# parcel summaries - 2020 and 2025 NP for interpolating 2023 developed acres
parcel_NP_2020 = pd.read_csv(
    pathlib.Path(NP_DIR) / "core_summaries" / f"{NP_RUNID}_parcel_summary_2020.csv",
    usecols=["parcel_id", "tothh", "totemp", "non_residential_sqft", "residential_units"]
)
parcel_NP_2025 = pd.read_csv(
    pathlib.Path(NP_DIR) / "core_summaries" / f"{NP_RUNID}_parcel_summary_2025.csv",
    usecols=["parcel_id", "tothh", "totemp", "non_residential_sqft", "residential_units"]
)

# parcel summaries - 2035 FBP, 2050 NP/FBP, for calculating 2035 FBP/2050 FBP developed acres, 2035/2050 growth patterns (including density)
parcel_FBP_2050 = pd.read_csv(
    pathlib.Path(FBP_DIR) / "core_summaries" / f"{FBP_RUNID}_parcel_summary_2050.csv",
    usecols=["parcel_id", "tothh", "totemp", "non_residential_sqft", "residential_units"]
)
parcel_NP_2050 = pd.read_csv(
    pathlib.Path(NP_DIR) / "core_summaries" / f"{NP_RUNID}_parcel_summary_2050.csv",
    usecols=["parcel_id", "tothh", "totemp", "non_residential_sqft", "residential_units"]
)

parcel_FBP_2035 = pd.read_csv(
    pathlib.Path(FBP_DIR) / "core_summaries" / f"{FBP_RUNID}_parcel_summary_2035.csv",
    usecols=["parcel_id", "tothh", "totemp", "non_residential_sqft", "residential_units"]
)

data_dict = {
    "2020 No Project": parcel_NP_2020,
    "2025 No Project": parcel_NP_2025,
    "2035 Final Blueprint": parcel_FBP_2035,
    "2050 Final Blueprint": parcel_FBP_2050,
    "2050 No Project": parcel_NP_2050
}

In [4]:
# PBA50+ version of parcel geographies crosswalk
parcels_geographies = pd.read_csv(
    pathlib.Path(M_DRIVE) / "urban_modeling" / "baus" / "BAUS Inputs" / "basis_inputs" / "crosswalks" / "fbp_urbansim_parcel_classes_ot50pct_feb25_rwc_toc_may_9_2025.csv"
)
print(list(parcels_geographies.shape))
print(parcels_geographies['parcel_id'].nunique())
display(parcels_geographies.head())

# PBA50 version of the data which includes ACRES
parcels_geographies_pba50 = pd.read_csv(
    pathlib.Path(M_DRIVE) / "urban_modeling" / "baus" / "BAUS Inputs" / "basis_inputs" / "crosswalks" / "2021_02_25_parcels_geography.csv"
)
print(list(parcels_geographies_pba50))
print(parcels_geographies_pba50['PARCEL_ID'].nunique())
display(parcels_geographies_pba50.head())

parcels_geographies = parcels_geographies.merge(
    parcels_geographies_pba50[["PARCEL_ID", "ACRES"]].rename(columns={"PARCEL_ID": "parcel_id", "ACRES": "acres"}),
    on="parcel_id",
    how="left"
)
print(parcels_geographies.shape)
print(parcels_geographies['parcel_id'].nunique())


[1956207, 14]
1956207


,Unnamed: 0,parcel_id,gg_id,pda_id,ppa_id,tra_id,hra_id,ugb_id,tpp_id,dis_id,exp2020_id,exsfd_id,exd_id,toc_id
0,0,22.0,GG,Oakland - West Oakland,NaN,NaN,NaN,UGB,brt3,NaN,in,NaN,NaN,NaN
1,1,58.0,GG,NaN,PPA,NaN,NaN,UGB,NaN,NaN,in,NaN,NaN,NaN
2,2,128.0,GG,Oakland - West Oakland,NaN,NaN,NaN,UGB,bart1,NaN,in,NaN,NaN,toc
3,3,217.0,GG,Oakland - Downtown & Jack London Square,NaN,tra_4,NaN,UGB,brt1,NaN,in,NaN,NaN,toc
4,4,358.0,GG,Oakland - Downtown & Jack London Square,NaN,tra_4,NaN,UGB,lrt2,DIS,in,NaN,NaN,toc


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\3962064122.py:10: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  parcels_geographies_pba50 = pd.read_csv(


['Unnamed: 0', 'PARCEL_ID', 'geom_id', 'jurisdiction_id', 'juris_name_full', 'juris_id', 'juris', 'ACRES', 'tpp_id', 'exp_id', 'opp_id', 'perffoot', 'perfarea', 'urbanized', 'hra_id', 'cat_id', 'exp2020_id', 'exsfd_id', 'gg_id', 'pda_id', 'tra_id', 'sesit_id', 'ppa_id', 'fbp_exp2020_id', 'fbp_exsfd_id', 'zoningmodcat', 'chcat', 'pda_id_pba50_fb.1', 'coc_id', 'eir_exp2020_id', 'ex_res_bldg', 'nodev']
1956208


,Unnamed: 0,PARCEL_ID,geom_id,jurisdiction_id,juris_name_full,juris_id,juris,ACRES,tpp_id,exp_id,...,ppa_id,fbp_exp2020_id,fbp_exsfd_id,zoningmodcat,chcat,pda_id_pba50_fb.1,coc_id,eir_exp2020_id,ex_res_bldg,nodev
0,0,229116,10305106092872,41992,livermore,livr,livermore,3.360520,NaN,NaN,...,NaN,in,NaN,livermoreNANAHRADISNAinNA,NANAHRADISNAin,NaN,NaN,in,NaN,0.0
1,1,244166,11107351665227,41992,livermore,livr,livermore,1.294423,NaN,NaN,...,NaN,in,NaN,livermoreNAtra3DISNAinNA,NAtra3DISNAin,NaN,NaN,in,NaN,0.0
2,2,202378,11030175960628,33000,hayward,hayw,hayward,14.993605,NaN,NaN,...,NaN,in,NaN,haywardNANANANAinNA,NANANANAin,NaN,NaN,in,res,0.0
3,3,2004420,6381677629073,97,unincorporated_sonoma,uson,unincorporated_sonoma,316.247146,NaN,NaN,...,NaN,out,NaN,unincorporated_sonomaNANAHRADISNAoutNA,NANAHRADISNAout,NaN,NaN,out,NaN,0.0
4,4,340332,314875459798,26000,fremont,frem,fremont,0.621275,b1,NaN,...,NaN,in,NaN,fremontNANAHRADISNAinNA,NANAHRADISNAin,NaN,NaN,in,NaN,1.0


(1956207, 15)
1956207


In [5]:
# calculate total acres by growth geography types

PARCEL_AREA_FILTERS = {
    'HRA'       : lambda df: df['hra_id'] == 'HRA',
    'TRA'       : lambda df: df['tra_id'].isin(['tra_1', 'tra_2', 'tra_3', 'tra_4', 'tra_5']),
    'GG'        : lambda df: df['gg_id'] == 'GG',
    'PDA'       : lambda df: pd.notna(df['pda_id']),
    'Region'    : None,
    'TOC'       : lambda df: df['toc_id'] == 'toc'
}

geograhpies_area = pd.DataFrame(columns=['area', 'acres'])
for area, filter_condition in PARCEL_AREA_FILTERS.items():
    if callable(filter_condition):  # Check if the filter is a function
        df_area = parcels_geographies.loc[filter_condition(parcels_geographies)]
    elif filter_condition == None:
        df_area = parcels_geographies
    print("area={} df_area len={:,}".format(area, len(df_area)))
    geograhpies_area.loc[len(geograhpies_area)] = [area, df_area['acres'].sum()]
display(geograhpies_area)

area=HRA df_area len=785,585
area=TRA df_area len=518,656
area=GG df_area len=676,221
area=PDA df_area len=320,472
area=Region df_area len=1,956,207
area=TOC df_area len=404,292


,area,acres
0,HRA,6.407189e+05
1,TRA,1.224186e+05
2,GG,2.173730e+05
3,PDA,9.780761e+04
4,Region,4.488103e+06
5,TOC,1.019195e+05


### Calculate density (hh/acre) for 2023/2035/2050 FBP and 2050 NP 

In [6]:
# Start with existing growth patterns by geography summary
growth_patterns_metrics = pd.read_csv(
    pathlib.Path(BOX_DIR) / "Plan Bay Area 2050+" / "Performance and Equity" / "Plan Performance" / "Equity_Performance_Metrics" / "Final_Blueprint" / "metrics_growthPattern_geographies_2050.csv",
    usecols=['modelrun_id', 'modelrun_alias', 'area', 'total_households', 'total_jobs']
)
display(growth_patterns_metrics.head())

print(growth_patterns_metrics['modelrun_id'].unique())
print(growth_patterns_metrics['modelrun_alias'].unique())
print(growth_patterns_metrics['area'].unique())

growth_patterns_metrics = growth_patterns_metrics.loc[
    growth_patterns_metrics['modelrun_id'].isin(
        ['PBA50Plus_NoProject/PBA50Plus_NoProject_v38', 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v65']) & \
    growth_patterns_metrics['modelrun_alias'].isin(['2023 No Project', '2050 Final Blueprint','2050 No Project']) & \
    growth_patterns_metrics['area'].isin(['HRA', 'TRA', 'GG', 'PDA', 'Region', 'TOC'])]

,modelrun_id,modelrun_alias,area,total_households,total_jobs
0,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,HRA,1138568.6,1384330.6
1,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,TRA,1112576.2,1972134.8
2,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,HRAandTRA,429183.0,584953.0
3,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,GG,1373649.0,2869387.4
4,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,nonGG,1485031.6,1233077.0


['PBA50Plus_NoProject/PBA50Plus_NoProject_v38'
 'PBA50Plus_DraftBlueprint/PBA50Plus_Draft_Blueprint_v8_znupd_nodevfix'
 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v65'
 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v65_exog_p92'
 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v69'
 'PBA50Plus_FinalBlueprint/PBA50Plus_Final_Blueprint_v69_no_ubi'
 'PBA50Plus_EIR/PBA50Plus_EIR_Alt2_Var4_v1']
['2023 No Project' '2050 No Project' '2023 Draft Blueprint'
 '2050 Draft Blueprint' '2023 Final Blueprint' '2050 Final Blueprint'
 '2023 EIR Alt1' '2050 EIR Alt1' '2023 EIR Alt2' '2050 EIR Alt2']
['HRA' 'TRA' 'HRAandTRA' 'GG' 'nonGG' 'PBA50GG' 'PBA50nonGG' 'GG_nonPDA'
 'PDA' 'EPC_18' 'nonEPC_18' 'EPC_22' 'nonEPC_22' 'PPA' 'Region' 'TOC'
 'nonTOC']


In [7]:
growth_patterns_metrics

,modelrun_id,modelrun_alias,area,total_households,total_jobs
0,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,HRA,1138568.6,1384330.6
1,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,TRA,1112576.2,1972134.8
3,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,GG,1373649.0,2869387.4
8,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,PDA,818111.0,1970302.4
14,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,Region,2858680.6,4102464.4
15,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,TOC,912852.8,1791880.0
17,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,HRA,1434914.0,1765849.0
18,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,TRA,1576257.0,2670101.0
20,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,GG,1995395.0,3863497.0
25,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,PDA,1322970.0,2686201.0


In [8]:
# also summarize 2035 and append to growth_patterns_metrics
parcel_FBP_2035 = parcel_FBP_2035.merge(
    parcels_geographies,
    on="parcel_id",
    how="left"
)
    
for area, filter_condition in PARCEL_AREA_FILTERS.items():

    if callable(filter_condition):  # Check if the filter is a function
        df_FBP_2035_area = parcel_FBP_2035.loc[filter_condition(parcel_FBP_2035)]
    elif filter_condition == None:
        df_FBP_2035_area = parcel_FBP_2035
    
    growth_patterns_metrics = growth_patterns_metrics.append({
        'modelrun_id': FBP_RUNID,
        'modelrun_alias': '2035 Final Blueprint',
        'area': area,
        'total_households': df_FBP_2035_area['tothh'].sum(),
        'total_jobs': df_FBP_2035_area['totemp'].sum()
    }, ignore_index=True)


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\68708230.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  growth_patterns_metrics = growth_patterns_metrics.append({
C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\68708230.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  growth_patterns_metrics = growth_patterns_metrics.append({
C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\68708230.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  growth_patterns_metrics = growth_patterns_metrics.append({
C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\68708230.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  growth_patterns_met

In [9]:
# merge growth_patterns_metrics with geographies_area to calculate household density by area and modelrun_alias
growth_patterns_metrics_with_area = growth_patterns_metrics.merge(
    geograhpies_area,
    on="area",
    how="left"
)

growth_patterns_metrics_with_area['hh_density'] = growth_patterns_metrics_with_area['total_households'] / growth_patterns_metrics_with_area['acres']
display(growth_patterns_metrics_with_area)
growth_patterns_metrics_with_area.to_csv(
    pathlib.Path(BOX_DIR) / "Plan Bay Area 2050+" / "Performance and Equity" / "Plan Performance" / "Equity_Performance_Metrics" / "Final_Blueprint" / "CARBmetrics_growth_patterns.csv",
    index=False)


,modelrun_id,modelrun_alias,area,total_households,total_jobs,acres,hh_density
0,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,HRA,1138568.6,1384330.6,6.407189e+05,1.777017
1,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,TRA,1112576.2,1972134.8,1.224186e+05,9.088296
2,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,GG,1373649.0,2869387.4,2.173730e+05,6.319317
3,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,PDA,818111.0,1970302.4,9.780761e+04,8.364493
4,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,Region,2858680.6,4102464.4,4.488103e+06,0.636946
5,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2023 No Project,TOC,912852.8,1791880.0,1.019195e+05,8.956606
6,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,HRA,1434914.0,1765849.0,6.407189e+05,2.239538
7,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,TRA,1576257.0,2670101.0,1.224186e+05,12.875964
8,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,GG,1995395.0,3863497.0,2.173730e+05,9.179590
9,PBA50Plus_NoProject/PBA50Plus_NoProject_v38,2050 No Project,PDA,1322970.0,2686201.0,9.780761e+04,13.526248


### Calculate total developed acres

In [10]:
# loop through each model run, merge with geographies to get acres,
# calculate developed acres based on either has non-residential sqft or residential units,
# and summarize total developed acres by model run

developed_acres_df = pd.DataFrame(columns=['modelrun_alias', 'developed_acres'])

for run_alias in data_dict.keys():
    df = data_dict[run_alias].merge(
        parcels_geographies[['parcel_id', 'acres']],
        on="parcel_id",
        how="left"
    )
    df['developed_acres'] = df.apply(lambda row: row['acres'] if (row['non_residential_sqft'] > 0 or row['residential_units'] > 0) else 0, axis=1)
    print(f"{run_alias}, total developed acres: {df['developed_acres'].sum():,}")
    developed_acres_df = developed_acres_df.append({
        'modelrun_alias': run_alias,
        'developed_acres': df['developed_acres'].sum()
    }, ignore_index=True)

display(developed_acres_df)

2020 No Project, total developed acres: 1,330,118.35585367


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\1076474140.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  developed_acres_df = developed_acres_df.append({


2025 No Project, total developed acres: 1,336,165.6478650158


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\1076474140.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  developed_acres_df = developed_acres_df.append({


2035 Final Blueprint, total developed acres: 1,339,553.0425801852


...

2050 Final Blueprint, total developed acres: 1,343,040.6332818698


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\1076474140.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  developed_acres_df = developed_acres_df.append({


2050 No Project, total developed acres: 1,348,417.5423132377


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\1076474140.py:15: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  developed_acres_df = developed_acres_df.append({


,modelrun_alias,developed_acres
0,2020 No Project,1.330118e+06
1,2025 No Project,1.336166e+06
2,2035 Final Blueprint,1.339553e+06
3,2050 Final Blueprint,1.343041e+06
4,2050 No Project,1.348418e+06


In [11]:
# interpolate 2023 developed acres based on 2020 and 2025 NP data, and append to developed_acres_df

t1, t2 = 2020, 2025
acres_2020 = developed_acres_df.loc[developed_acres_df['modelrun_alias'] == '2020 No Project', 'developed_acres'].sum()
acres_2025 = developed_acres_df.loc[developed_acres_df['modelrun_alias'] == '2025 No Project', 'developed_acres'].sum()
acres_2023 = acres_2020 + (acres_2025 - acres_2020) * (2023 - t1) / (t2 - t1)
print(f"Interpolated 2023 developed acres: {acres_2023:,.2f}")

developed_acres_df = developed_acres_df.append({
    'modelrun_alias': '2023 No Project',
    'developed_acres': acres_2023
}, ignore_index=True)

display(developed_acres_df)

developed_acres_df.to_csv(
    pathlib.Path(BOX_DIR) / "Plan Bay Area 2050+" / "Performance and Equity" / "Plan Performance" / "Equity_Performance_Metrics" / "Final_Blueprint" / "CARBmetrics_developed_acres.csv",
    index=False)

Interpolated 2023 developed acres: 1,333,746.73


C:\Users\ywang\AppData\Local\Temp\ipykernel_32936\4155983888.py:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  developed_acres_df = developed_acres_df.append({


,modelrun_alias,developed_acres
0,2020 No Project,1.330118e+06
1,2025 No Project,1.336166e+06
2,2035 Final Blueprint,1.339553e+06
3,2050 Final Blueprint,1.343041e+06
4,2050 No Project,1.348418e+06
5,2023 No Project,1.333747e+06
